In [ ]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

model = GenericFakeChatModel(
    messages=iter([
        "hello",
        "world"
    ])
)

print(model.invoke("anything").content)
print(model.invoke("anything").content)

hello
world


In [3]:
from langchain_core.messages import AIMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

fake_model = GenericFakeChatModel(
    messages=iter([
        AIMessage(
            content="",
            tool_calls=[
                {
                    "id": "call_1",
                    "name": "foo",
                    "args": {"bar": "baz"},
                }
            ]
        ),
        "final answer"
    ])
)

print(fake_model.invoke("anything"))

content='' additional_kwargs={} response_metadata={} id='lc_run--019b681e-c6f3-73e1-b878-8e7f3fce4a6a-0' tool_calls=[{'name': 'foo', 'args': {'bar': 'baz'}, 'id': 'call_1', 'type': 'tool_call'}]


In [6]:
from langchain.tools import tool

@tool
def fake_weather(city: str) -> str:
    """Fake weather tool for testing."""
    return f"FAKE_WEATHER({city})"

from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel

fake_model = GenericFakeChatModel(
    messages=iter([
        AIMessage(
            content="",
            tool_calls=[
                {
                    "id": "call_1",
                    "name": "fake_weather",
                    "args": {"city": "Beijing"},
                }
            ]
        ),
        "Final answer"
    ])
)

from langchain.agents import create_agent

agent = create_agent(
    model=fake_model,
    tools=[fake_weather],
)

result = agent.invoke({"messages": [HumanMessage("What's the weather like in Beijing?")]})

NotImplementedError: 

In [8]:
from langchain.tools import tool

@tool
def get_weather(city: str):
    """Get weather information for a city."""
    return f"It's 75 degrees and sunny in {city}."

from langchain_community.chat_models import ChatTongyi
import os

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")

from langchain.agents import create_agent
agent = create_agent(model, tools=[get_weather])

In [16]:
from agentevals.trajectory.match import create_trajectory_match_evaluator

evaluator = create_trajectory_match_evaluator(  
    trajectory_match_mode="strict",  
)  

In [17]:
from langchain.messages import HumanMessage, AIMessage, ToolMessage

def test_weather_tool_called_strict():
    result = agent.invoke({
        "messages": [HumanMessage(content="What's the weather in San Francisco?")]
    })

    reference_trajectory = [
        HumanMessage(content="What's the weather in San Francisco?"),
        AIMessage(content="", tool_calls=[
            {"id": "call_1", "name": "get_weather", "args": {"city": "San Francisco"}}
        ]),
        ToolMessage(content="It's 75 degrees and sunny in San Francisco.", tool_call_id="call_1"),
        AIMessage(content="The weather in San Francisco is 75 degrees and sunny."),
    ]

    evaluation = evaluator(
        outputs=result["messages"],
        reference_outputs=reference_trajectory
    )
    # {
    #     'key': 'trajectory_strict_match',
    #     'score': True,
    #     'comment': None,
    # }
    assert evaluation["score"] is True

In [ ]:
import json
from agentevals.trajectory.llm import TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,create_trajectory_llm_as_judge, TRAJECTORY_ACCURACY_PROMPT
from langchain_community.chat_models import ChatTongyi
import os

model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")
evaluator = create_trajectory_llm_as_judge(
  prompt=TRAJECTORY_ACCURACY_PROMPT,
  model=model
)
outputs = [
    {"role": "user", "content": "What is the weather in SF?"},
    {
        "role": "assistant",
        "content": "",
        "tool_calls": [
            {
                "function": {
                    "name": "get_weather",
                    "arguments": json.dumps({"city": "SF"}),
                }
            }
        ],
    },
    {"role": "tool", "content": "It's 80 degrees and sunny in SF."},
    {"role": "assistant", "content": "The weather in SF is 80 degrees and sunny."},
]
eval_result = evaluator(
    outputs=outputs,
)

print(eval_result)

In [ ]:
{
    'key': 'trajectory_accuracy',
    'score': True,
    'comment': 'The provided agent trajectory is reasonable...'
}